# Nemotron v7.2 — Training Notebook (Bundled Standalone)

Same stack as `nemotron_v7_training.ipynb` (DoRA + rsLoRA + PiSSA + LoRA+ +
stratified batching), packaged as a standalone copy alongside its matching
submission notebook.

**Outputs:** `adapter.zip` + per-epoch `checkpoints/adapter_epoch_XX.zip`

### Eval-server contract (vLLM) this notebook respects
| Eval parameter | Value |
|---|---|
| `max_lora_rank` | 32 |
| `max_tokens` | 7680 |
| `max_model_len` | 8192 |
| `temperature` | 0.0 (greedy) |
| `top_p` | 1.0 |
| `max_num_seqs` | 64 |
| `gpu_memory_utilization` | 0.85 |

### Adapter stack
- **DoRA** — magnitude/direction split (+1–3%)
- **rsLoRA** — α/√r scaling, stable at r=32 (+0.5–1.5%)
- **PiSSA init** — SVD warm-start (+1–2%, 2–3× faster convergence)
- **LoRA+** — B trains 16× faster than A (+0.5–1%)
- **Stratified batching** — one domain per effective batch (reduces gradient noise)


In [ ]:
# ============================================================
# 1. OFFLINE DEPENDENCY INSTALLATION
# ============================================================
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = pth_file.parent / relpath
            if rel_pack_path.exists():
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir  = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)

resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl", "peft"
    ])
    print("Offline packages installed.")

# wandb — must be set BEFORE import so it never tries to connect
os.environ["WANDB_MODE"] = "offline"

# Try installing wandb: offline packages first, then pip (if internet exists)
WANDB_AVAILABLE = False
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "wandb"
    ])
    WANDB_AVAILABLE = True
    print("wandb installed (offline).")
except Exception:
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "wandb"
        ])
        WANDB_AVAILABLE = True
        print("wandb installed (online).")
    except Exception:
        print("wandb not available — training will continue without W&B logging.")

sys.path.append(target_dir)
resolve_python_path(target_dir)

In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import stat, shutil, zipfile, time, json, re
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
from collections import Counter

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"W&B     : {'offline mode' if WANDB_AVAILABLE else 'disabled'}")

In [ ]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE (no internet needed)
# ============================================================
# W&B runs in offline mode: all metrics are saved locally to
# /kaggle/working/wandb/. After training, the run directory is
# zipped so you can download it and sync from your local machine:
#
#   wandb sync /path/to/wandb/offline-run-XXXXXXXX-XXXXXXXX
#
# This gives you full dashboard access (loss curves, LR schedule,
# grad norms, system metrics) — just delayed until you sync.

WANDB_PROJECT  = "nemotron-v7"
WANDB_RUN_NAME = "v7-lora-r32-a64-lr2e5-3ep"
WANDB_DIR      = "/kaggle/working"   # wandb creates ./wandb/ under this

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=WANDB_DIR,
        config={
            "model": "Nemotron-3-Nano-30B-A3B",
            "lora_rank": 32,
            "lora_alpha": 64,
            "learning_rate": 1e-4,
            "effective_lr": 4e-5,
            "num_epochs": 3,
            "batch_size": 1,
            "grad_accum": 8,
            "effective_batch": 16,
            "max_seq_len": 8192,
            "lora_targets": "q,k,v,o,in,out,up,down,lm_head",
            "lora_dropout": 0.0,
            "warmup_steps": 50,
            "scheduler": "cosine",
            "packing": True,
            "bf16": True,
        },
        tags=["nemotron", "lora", "v7", "sft"],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
    print(f"After training, download wandb_logs.zip and run:")
    print(f"  wandb sync <run_dir>")
else:
    print("W&B not available — skipping init. Training metrics logged to stdout only.")

In [ ]:
# ============================================================
# 3. TRITON FIXES (rmsnorm + ptxas-blackwell binary copy) — DEFENSIVE
# ============================================================
# Two fixes:
#  (a) rmsnorm_fn replaced with pure PyTorch (mamba_ssm Triton kernel
#      fails on some GPU configs)
#  (b) ptxas-blackwell binary lives in a read-only mount without +x;
#      copy it to /tmp and chmod, then redirect Triton env vars and
#      bust caches. NO shutil.copytree — that path doesn't always exist.

import glob

# (a) rmsnorm_fn patch
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# (b) Find ptxas-blackwell anywhere — handle both hyphen and underscore paths
candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
    + glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cuda_nvcc/bin/ptxas",
                recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) \
   or (candidates[0] if candidates else None)

if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst,
             os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    # Point all Triton env vars to the writable copy
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = dst

    # Bust Triton caches so the new path is honored
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        try: nv_compiler.get_ptxas_version.cache_clear()
        except AttributeError: pass
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
        from triton import knobs as triton_knobs
        for attr in ("ptxas", "ptxas_blackwell"):
            triton_knobs.nvidia.__dict__.pop(attr, None)
    except Exception as e:
        print(f"Triton cache clear warning: {e}")

    print(f"[ok] ptxas binary -> {dst}  (copied from {src})")
else:
    print("[warn] no ptxas binary found anywhere — Mamba kernel will crash")


In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7.2 (DoRA + rsLoRA + PiSSA + LoRA+)
# ============================================================
LORA_RANK           = 32          # eval-server cap
LORA_ALPHA          = 64          # 2:1 ratio
MAX_SEQ_LEN         = 7680        # eval-server max_tokens
NUM_EPOCHS          = 3
BATCH_SIZE          = 4
GRAD_ACCUM          = 8           # effective batch = 32
LR                  = 2e-5        # lowered for rsLoRA (α/√r inflates effective LR)
WARMUP_STEPS        = 50
SAVE_EVERY_N_EPOCHS = 1

USE_DORA       = True
USE_RSLORA     = True
PISSA_INIT     = "pissa_niter_4"
LORAPLUS_RATIO = 16

USE_STRATIFIED_BATCHING = True

MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR  = "/kaggle/working/adapter"
CKPT_DIR    = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

OUR_DATA_PATHS = [
    "/kaggle/input/nemotron-cot-v5/train_cot_v5_merged.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v5_merged.jsonl",
    "/kaggle/input/nemotron-cot-v4/train_cot_v4_real.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v4_real.jsonl",
]
EXTERNAL_CSV_PATHS = [
    "/kaggle/input/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv",
]

assert LORA_RANK   <= 32,   f"LORA_RANK={LORA_RANK} exceeds eval cap 32"
assert MAX_SEQ_LEN <= 7680, f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds eval cap 7680"
if USE_RSLORA:
    assert LR <= 5e-5, f"LR={LR} too high with rsLoRA"

print(f"Rank/alpha : {LORA_RANK}/{LORA_ALPHA}  | seqlen {MAX_SEQ_LEN}  | LR {LR:.1e}")
print(f"Epochs     : {NUM_EPOCHS}  | batch {BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")
print(f"Adapter    : DoRA={USE_DORA} rsLoRA={USE_RSLORA} init={PISSA_INIT} LoRA+={LORAPLUS_RATIO}")
print(f"Stratified : {USE_STRATIFIED_BATCHING}")


In [ ]:
# ============================================================
# 5. CALLBACKS — progress bar + per-epoch checkpoint zip
# ============================================================
class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training", dynamic_ncols=True)
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.pbar is None or not logs:
            return
        self.pbar.update(state.global_step - self.pbar.n)
        msg = []
        if "loss" in logs:          msg.append(f"loss={logs['loss']:.4f}")
        if "learning_rate" in logs: msg.append(f"lr={logs['learning_rate']:.2e}")
        if msg:
            self.pbar.set_postfix_str(" ".join(msg))

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar: self.pbar.close()


class CheckpointZipCallback(TrainerCallback):
    def __init__(self, ckpt_dir, output_dir, every_n=1, model_path=None,
                 pissa_init=None):
        self.ckpt_dir    = ckpt_dir
        self.output_dir  = output_dir
        self.every_n     = every_n
        self.model_path  = model_path
        self.pissa_init  = pissa_init
        self.epoch_losses = {}

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.epoch_losses[int(state.epoch)] = logs["loss"]

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)
        if epoch == 0 or epoch % self.every_n != 0:
            return
        epoch_dir = os.path.join(self.ckpt_dir, f"adapter_epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)
        save_kwargs = {}
        if self.pissa_init and self.pissa_init.startswith("pissa") and self.model_path:
            save_kwargs["path_initial_model_for_weight_conversion"] = self.model_path
        model.save_pretrained(epoch_dir, **save_kwargs)

        cfg_path = os.path.join(epoch_dir, "adapter_config.json")
        if os.path.exists(cfg_path):
            with open(cfg_path) as f: cfg = json.load(f)
            cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
            with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

        zip_path = os.path.join(self.ckpt_dir, f"adapter_epoch_{epoch:02d}.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in os.listdir(epoch_dir):
                fp = os.path.join(epoch_dir, fname)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fname)
        sz = os.path.getsize(zip_path) / 1024 / 1024
        loss = self.epoch_losses.get(epoch, float('nan'))
        print(f"[ckpt] epoch {epoch:2d} | loss={loss:.4f} | saved {zip_path} ({sz:.1f} MB)")

    def print_summary(self):
        if not self.epoch_losses:
            return
        best = min(self.epoch_losses.items(), key=lambda x: x[1])
        print(f"\n=== Checkpoint summary ===")
        for e, l in sorted(self.epoch_losses.items()):
            tag = " <-- BEST" if e == best[0] else ""
            print(f"  epoch {e:2d}  loss={l:.4f}{tag}")

ckpt_callback = CheckpointZipCallback(
    CKPT_DIR, OUTPUT_DIR, every_n=SAVE_EVERY_N_EPOCHS,
    model_path=MODEL_PATH, pissa_init=PISSA_INIT,
)


In [ ]:
# ============================================================
# 6. LOAD DATASET
# ============================================================
our_data = []
ext_csv_path = None

for path in OUR_DATA_PATHS:
    if os.path.exists(path):
        print(f"Found JSONL: {path}")
        with open(path) as f:
            for line in f:
                our_data.append(json.loads(line))
        break

for path in EXTERNAL_CSV_PATHS:
    if os.path.exists(path):
        ext_csv_path = path
        print(f"Found CSV: {path}")
        break

if not our_data and not ext_csv_path:
    raise FileNotFoundError("No training data found")

print(f"JSONL rows: {len(our_data)}  |  CSV: {bool(ext_csv_path)}")


In [ ]:
# ============================================================
# 7. FORMAT + INFER CATEGORY LABELS
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EVAL_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

CAT_KEYWORDS = [
    ("bit_manipulation", ["binary", "bit", "8-bit", "byte", "XOR", "rotate"]),
    ("cipher",           ["cipher", "encrypt", "decrypt", "substitut"]),
    ("numeral",          ["roman", "numeral", "XVIII", "XXIV"]),
    ("unit_conversion",  ["convert", "unit", "meter", "mile", "kilogram"]),
    ("gravity",          ["gravity", "gravitational", "d = 0.5", "seconds"]),
    ("transformation",   ["rule", "@", "&", "transform", "operator"]),
]

def infer_category(prompt):
    p = prompt.lower()
    scores = {n: sum(1 for k in kws if k.lower() in p) for n, kws in CAT_KEYWORDS}
    scores = {k: v for k, v in scores.items() if v}
    return max(scores, key=scores.get) if scores else "other"

all_texts  = []
all_labels = []

if our_data:
    for ex in our_data:
        messages = [m for m in ex['messages'] if m['role'] != 'system']
        user_content = messages[0]['content'] if messages and messages[0]['role'] == 'user' else ''
        asst = messages[-1]
        if '<think>' not in asst['content']:
            m = re.search(r'(\\boxed\{.*?\})\s*$', asst['content'])
            if m:
                reasoning = asst['content'][:m.start()].strip()
                messages[-1] = {'role': 'assistant',
                                'content': f"<think>\n{reasoning}\n</think>\n{m.group(1)}"}
        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=False)
        except Exception:
            text = (f"<|im_start|>user\n{messages[0]['content']}<|im_end|>\n"
                    f"<|im_start|>assistant\n{messages[-1]['content']}<|im_end|>")
        all_texts.append(text)
        all_labels.append(ex.get('category') or ex.get('label')
                          or infer_category(user_content))

if ext_csv_path:
    import pandas as pd
    df = pd.read_csv(ext_csv_path)
    seen = {ex['messages'][0]['content'][:200] for ex in our_data
            if our_data and ex['messages'][0]['role'] == 'user'}
    for _, row in df.iterrows():
        prompt = str(row.get('prompt', ''))
        if prompt[:200] in seen: continue
        answer = str(row.get('answer', ''))
        cot    = str(row.get('generated_cot', ''))
        if not prompt.strip() or not answer.strip(): continue
        user_msg = prompt + EVAL_SUFFIX
        asst_msg = f"<think>\n{cot}\n</think>\n\\boxed{{{answer}}}"
        msgs = [{'role': 'user', 'content': user_msg},
                {'role': 'assistant', 'content': asst_msg}]
        try:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=False)
        except Exception:
            text = (f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                    f"<|im_start|>assistant\n{asst_msg}<|im_end|>")
        all_texts.append(text)
        lbl = str(row.get('category', '') or row.get('type', '')).strip().lower()
        all_labels.append(lbl if lbl and lbl != 'nan' else infer_category(prompt))

hf_dataset = Dataset.from_dict({'text': all_texts, 'label': all_labels})
print(f"\nTotal: {len(hf_dataset)} examples")
print("Category distribution:")
for n, c in Counter(all_labels).most_common():
    print(f"  {n:20s} {c:5d}  ({100*c/len(all_labels):.1f}%)")


In [ ]:
# ============================================================
# 8. DROP OVERSIZED SAMPLES (> MAX_SEQ_LEN)
# ============================================================
before = len(hf_dataset)
def _len(ex):
    return {'tl': len(tokenizer(ex['text'], truncation=False,
                                 return_attention_mask=False)['input_ids'])}
hf_dataset = hf_dataset.map(_len, desc="Counting tokens")
hf_dataset = hf_dataset.filter(lambda x: x['tl'] <= MAX_SEQ_LEN, desc="Filtering")
hf_dataset = hf_dataset.remove_columns(['tl'])
print(f"Kept {len(hf_dataset)}/{before}")


In [ ]:
# ============================================================
# 9. LOAD MODEL (bf16)
# ============================================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print(f"Model loaded: {type(model).__name__}")


In [ ]:
# ============================================================
# 10. APPLY LoRA — DoRA + rsLoRA + PiSSA
# ============================================================
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj",
    "up_proj", "down_proj",
    "lm_head",
]

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=["embed_tokens"],
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    use_dora=USE_DORA,
    use_rslora=USE_RSLORA,
    init_lora_weights=PISSA_INIT,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ============================================================
# 11. TRAINING — Stratified SFT with per-epoch ckpt zip
# ============================================================
from torch.utils.data import Sampler
import random as _random

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM


def build_stratified_index_order(labels, chunk_size, seed=0):
    buckets = {}
    for i, lbl in enumerate(labels):
        buckets.setdefault(lbl, []).append(i)
    rng = _random.Random(seed)
    for lbl in buckets:
        rng.shuffle(buckets[lbl])
    order = []
    active = list(buckets.keys())
    rng.shuffle(active)
    while active:
        next_active = []
        for lbl in active:
            take = buckets[lbl][:chunk_size]
            buckets[lbl] = buckets[lbl][chunk_size:]
            order.extend(take)
            if buckets[lbl]:
                next_active.append(lbl)
        active = next_active
    return order


class PrecomputedOrderSampler(Sampler):
    def __init__(self, labels, chunk_size, base_seed=1337):
        self.labels = list(labels)
        self.chunk_size = chunk_size
        self.base_seed = base_seed
        self.epoch = 0
        self._order = build_stratified_index_order(self.labels, chunk_size, base_seed)
    def set_epoch(self, epoch):
        self.epoch = epoch
        self._order = build_stratified_index_order(self.labels, self.chunk_size,
                                                   self.base_seed + epoch)
    def __iter__(self): return iter(self._order)
    def __len__(self): return len(self.labels)


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_labels=None, chunk_size=16, **kwargs):
        self._strat_labels = stratified_labels
        self._strat_chunk  = chunk_size
        super().__init__(*args, **kwargs)
    def _get_train_sampler(self, *args, **kwargs):
        if self._strat_labels is None:
            return super()._get_train_sampler(*args, **kwargs)
        return PrecomputedOrderSampler(self._strat_labels, self._strat_chunk)


training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="no",
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    remove_unused_columns=False,
    loraplus_lr_ratio=LORAPLUS_RATIO,
)

labels_for_sampler = list(hf_dataset['label']) if USE_STRATIFIED_BATCHING else None

if USE_STRATIFIED_BATCHING:
    print(f"Stratified batching: chunk={EFFECTIVE_BATCH}")
    trainer = StratifiedSFTTrainer(
        model=model, train_dataset=hf_dataset, processing_class=tokenizer,
        args=training_args, callbacks=[LiveProgressCallback(), ckpt_callback],
        stratified_labels=labels_for_sampler, chunk_size=EFFECTIVE_BATCH,
    )
else:
    trainer = SFTTrainer(
        model=model, train_dataset=hf_dataset, processing_class=tokenizer,
        args=training_args, callbacks=[LiveProgressCallback(), ckpt_callback],
    )

print(f"Starting training: {len(hf_dataset)} samples, {NUM_EPOCHS} epochs, LR={LR:.1e}")
t0 = time.time()
trainer.train()
print(f"Done in {(time.time()-t0)/3600:.2f} hrs")
ckpt_callback.print_summary()


In [ ]:
# ============================================================
# 12. SAVE FINAL ADAPTER (PiSSA-aware conversion)
# ============================================================
save_kwargs = {}
if PISSA_INIT and PISSA_INIT.startswith("pissa"):
    save_kwargs["path_initial_model_for_weight_conversion"] = MODEL_PATH
    print("Saving with PiSSA -> standard LoRA conversion")

trainer.model.save_pretrained(OUTPUT_DIR, **save_kwargs)

cfg_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

print(f"r={cfg.get('r')} alpha={cfg.get('lora_alpha')}")
print(f"use_dora={cfg.get('use_dora')} use_rslora={cfg.get('use_rslora')}")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(fp):
        print(f"  {fn}  ({os.path.getsize(fp)/1e6:.2f} MB)")


In [ ]:
# ============================================================
# 13. ZIP FINAL ADAPTER
# ============================================================
ZIP_PATH = "/kaggle/working/adapter.zip"
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, fn)
        if os.path.isfile(fp):
            zf.write(fp, arcname=fn)

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"Zipped: {ZIP_PATH}  ({zip_mb:.1f} MB)")

print("\nPer-epoch checkpoints:")
for fn in sorted(os.listdir(CKPT_DIR)):
    if fn.endswith(".zip"):
        print(f"  {fn}  ({os.path.getsize(os.path.join(CKPT_DIR, fn))/1e6:.1f} MB)")

if WANDB_AVAILABLE:
    wandb.finish()
    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    if os.path.exists(wandb_dir):
        wz = "/kaggle/working/wandb_logs.zip"
        with zipfile.ZipFile(wz, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, _, files in os.walk(wandb_dir):
                for fn in files:
                    fp = os.path.join(root, fn)
                    zf.write(fp, arcname=os.path.relpath(fp, WANDB_DIR))
        print(f"W&B logs zipped: {wz}")
